In [1]:
import pandas as pd

In [2]:
import os

In [11]:
from openai import OpenAI

client = OpenAI()

In [28]:
import json

In [5]:
df = pd.read_json("../data-collection/scraped_data/scraped_articles.json")

In [6]:
df["content"].iloc[0]

'Ο Κ. Νικολακόπουλος γράφει στο blog του στο gazzetta για τους στόχους του Ολυμπιακού στο β΄ μισό της σεζόν.\n\nΗ νέα χρονιά ξεκινάει καλά για τον Ολυμπιακό. Όχι ιδανικά, αλλά καλά.\n\nΘα ήτανιδανικά τα πράγματα εάν στο πρωτάθλημα οΟλυμπιακόςδεν ξεπερνούσε τον χειρότερο εαυτό του σε γκέλες (εννέα χαμένοι βαθμοί, από τις ισοπαλίες με Παναιτωλικό, Λεβαδειακό εντός έδρας και Καλλιθέα εκτός έδρας, αλλά και την ήττα στην Τρίπολη, όπου όλοι περνάνε) και εάν στο Γιουρόπα Λιγκ κέρδιζε έστω ένα από τα τρία τελευταία παιχνίδια στα οποία έφερε ισοπαλίες (με Ρέϊντζερς, Στεάουα και Τβέντε). Εφόσον το είχε κάνει, το τρίποντο, θα ισοβαθμούσε τώρα με Ρέϊντζερς και Τότεναμ στην 8ηθέση της βαθμολογίας του μίνι πρωταθλήματος…\n\nΤέλος πάντων, ό,τι έγινε έγινε-όχι ότι δεν έχει σημασία, αλλά γιατί δεν αλλάζει. Πιο μεγάλη σημασία έχει τι θα γίνει. Κι ο Ολυμπιακός αυτή την στιγμή είναι σε τρεις στόχους, σημαντικούς, αλλά και δύσκολους.\n\nΠρώτος,η κατάκτηση του πρωταθλήματος. Βρίσκεται ναι μεν μόνος πρώτος σ

In [7]:
df["content"].str.contains("Μαριν").sum()

49

In [8]:
len(df["content"])

257

In [9]:
df["Marinakis_mention"] = df["content"].str.contains("Μαριν")

In [32]:

def get_sentiment_or_analysis(text: str) -> dict:
    """
    Send text to OpenAI for a 5-level sentiment analysis regarding Marinakis.
    Returns a dictionary with:
    {
      "mention_detected": bool,
      "sentiment": str,
      "explanation": str
    } 
    or a fallback if parsing fails.
    """

    prompt = f"""
You are an expert in Greek football journalism. 
Your task is to analyze a piece of text and determine how it portrays Evangelos Marinakis, the president of Olympiakos FC.

1. Classify the sentiment toward him with one of the following labels:
   - "HIGHLY NEGATIVE": Extremely critical or condemning tone
   - "NEGATIVE": Moderately critical or disapproving tone
   - "NEUTRAL": Factual, balanced, or neither positive nor negative
   - "POSITIVE": Moderately complimentary or supportive
   - "HIGHLY POSITIVE": Extremely praiseful or laudatory

3. Provide a short explanation for your classification.

4. Important: 
   - Output your final answer as valid JSON and nothing else.
   - Use the following JSON structure exactly:

{{
  "sentiment": "[ONE OF: HIGHLY NEGATIVE, NEGATIVE, NEUTRAL, POSITIVE, HIGHLY POSITIVE]",
  "explanation": "Short string explaining your reasoning"
}}

Text to analyze:
<<< TEXT START >>>
{text}
<<< TEXT END >>>
"""

    try:
        # Call the ChatCompletion endpoint
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
        )
        # Extract the model's message
        analysis_str = response.choices[0].message.content.strip()

        # Attempt to parse the JSON
        parsed_analysis = json.loads(analysis_str)

        return parsed_analysis

    except Exception as e:
        print(f"OpenAI API error or JSON parse error: {e}")
        # Fallback structure if something goes wrong
        return {
            "sentiment": "API_ERROR",
            "explanation": "Could not parse response."
        }


# Apply the helper function to rows where Marinakis is mentioned
responses = df.loc[df["Marinakis_mention"] == True, "content"].apply(get_sentiment_or_analysis)

print(responses)


0      {'sentiment': 'NEUTRAL', 'explanation': 'The t...
1      {'sentiment': 'POSITIVE', 'explanation': 'The ...
2      {'sentiment': 'NEUTRAL', 'explanation': 'The t...
4      {'sentiment': 'POSITIVE', 'explanation': 'The ...
12     {'sentiment': 'NEUTRAL', 'explanation': 'The t...
13     {'sentiment': 'NEUTRAL', 'explanation': 'The t...
14     {'sentiment': 'NEUTRAL', 'explanation': 'The t...
23     {'sentiment': 'POSITIVE', 'explanation': 'The ...
40     {'sentiment': 'NEUTRAL', 'explanation': 'The t...
42     {'sentiment': 'POSITIVE', 'explanation': 'The ...
45     {'sentiment': 'NEUTRAL', 'explanation': 'The t...
46     {'sentiment': 'NEUTRAL', 'explanation': 'The t...
56     {'sentiment': 'NEUTRAL', 'explanation': 'The t...
64     {'sentiment': 'NEUTRAL', 'explanation': 'The t...
69     {'sentiment': 'NEUTRAL', 'explanation': 'The t...
79     {'sentiment': 'NEUTRAL', 'explanation': 'The t...
81     {'sentiment': 'POSITIVE', 'explanation': 'The ...
92     {'sentiment': 'NEUTRAL',

In [41]:
df["sentiment"] = responses.map(lambda x: x["sentiment"])

In [43]:
df["sentiment"].value_counts()

NEUTRAL            27
POSITIVE           18
HIGHLY POSITIVE     3
NEGATIVE            1
Name: sentiment, dtype: int64

In [38]:
print(responses[214])

{'sentiment': 'HIGHLY POSITIVE', 'explanation': "The text portrays Evangelos Marinakis in a highly positive light, praising his decisions and leadership qualities, especially in the context of the team's success in European competitions. The text highlights his strategic thinking and ability to learn from mistakes, emphasizing his role in the team's achievements."}


In [37]:
df["content"][214]

'Ο Κ. Νικολακόπουλος γράφει στο blog του στο gazzetta για το μεγάλο μάθημα που άφησε η φετινή ευρωπαϊκή σεζόν στον Ολυμπιακό.\n\nΠριν τα δύο παιχνίδια του Ολυμπιακού με την Άστον Βίλα είχα γράψει μία κουβέντα, ότιοι παίκτες του Μεντιλίμπαρ έπρεπε να αφήσουν τα κόκκαλα τουςκαι στα δύο παιχνίδια, στο Βίλα Παρκ και στο Καραϊσκάκη, εάν ήθελαν να φτάσουν στον τελικό. Ακριβώς το ίδιο ίσχυε και στο παιχνίδι με τη Φιορεντίνα.\n\nΜη γελιόμαστε, ήταν ο μοναδικός τρόπος για να γίνονταν οι μεγάλες υπερβάσεις που έγιναν. Σίγουρα έπαιξαν ρόλο και το σχέδιο του προπονητή, αλλά και η ατομική ποιότητα των παικτών, αλλά χωρίς κατάθεση ψυχής και πάθους και πείσματος και αποφασιστικότητας, τίποτε δεν θα γίνονταν. Την δε Τετάρτη υπήρχε ένας παίκτης που έπαιξε πενήντα λεπτά και που έβγαλε ακριβώς αυτά τα στοιχεία, χωρίς ποτέ στην καριέρα του να ήταν τέτοιος παίκτης!\n\nΟ μπαλαδόροςκαι τεχνίτης και κλασάτος Γιόβετιτς μπήκε στο 72’ κι ως το τελευταίο λεπτό της παράτασης δεν σταμάτησε να τρέχει, να μαρκάρει, ν

In [34]:
print(responses[210])

{'sentiment': 'NEGATIVE', 'explanation': 'The text portrays Evangelos Marinakis in a critical tone, pointing out his decision-making process and hinting at potential negative consequences of his actions. The text also mentions potential issues related to player contracts and transfers, which contribute to an overall negative sentiment towards Marinakis.'}
